In [7]:
import pandas as pd
from sklearn.linear_model import Ridge, ElasticNet, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# 1. Đọc dữ liệu đã qua tiền xử lý
df = pd.read_csv("../data/processed/train_processed.csv")

# 2. Phân tách tập Train và Validation dựa trên cột 'split'
train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "validation"]

# 3. Loại bỏ các cột không phải là feature (Id, biến mục tiêu, nhãn split)
drop_cols = ["Id", "SalePrice", "SalePrice_log", "split"]
features = [col for col in df.columns if col not in drop_cols]

# 4. Chuẩn bị ma trận X và vector y (Sử dụng target đã được log1p)
X_train = train_df[features]
y_train = train_df["SalePrice_log"]

X_val = val_df[features]
y_val = val_df["SalePrice_log"]

# 5. Định nghĩa các mô hình Baseline
baseline_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression (alpha=1.0)": Ridge(alpha=1.0, random_state=42),
    "Random Forest Baseline": RandomForestRegressor(n_estimators=100, random_state=42),
}

# 6. Huấn luyện và Đánh giá
results = []
for name, model in baseline_models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    log_rmse = root_mean_squared_error(y_val, y_pred)
    log_mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)

    results.append(
        {
            "Model": name,
            "Validation log-RMSE": round(log_rmse, 4),
            "Validation log-MAE": round(log_mae, 4),
            "Validation R2": round(r2, 4),
        }
    )

results_df = pd.DataFrame(results)
print(results_df.to_markdown(index=False))

| Model                        |   Validation log-RMSE |   Validation log-MAE |   Validation R2 |
|:-----------------------------|----------------------:|---------------------:|----------------:|
| Linear Regression            |                0.1332 |               0.0955 |          0.9049 |
| Ridge Regression (alpha=1.0) |                0.1363 |               0.0973 |          0.9004 |
| Random Forest Baseline       |                0.1517 |               0.0992 |          0.8767 |


# Thiết lập Baseline Models

Notebook này thực hiện quá trình huấn luyện và đánh giá các mô hình học máy nhằm thiết lập cột mốc hiệu suất. Kết quả từ bước này sẽ được dùng làm tiêu chuẩn để đối chiếu và đánh giá sự cải thiện của cấu trúc mạng nơ-ron PyTorch MLP sau này.

## 1. Quy trình thực hiện

Các bước chính đã được triển khai trong mã nguồn bao gồm:
* **Nạp dữ liệu:** Đọc tập dữ liệu đã qua xử lý chuẩn hóa `train_processed.csv` từ bước tiền xử lý.
* **Phân chia dữ liệu:** Tách tập huấn luyện (train) và tập xác thực (validation) một cách triệt để dựa trên nhãn của cột `split` nhằm tránh rò rỉ dữ liệu (data leakage).
* **Chuẩn bị ma trận đầu vào:** Loại bỏ các cột định danh không có giá trị cho mô hình (`Id`, `split`, `SalePrice`). Sử dụng trực tiếp giá nhà đã biến đổi logarit (`SalePrice_log`) làm biến mục tiêu (target) để huấn luyện.
* **Huấn luyện mô hình:** Triển khai ba thuật toán cơ sở:
  * `Linear Regression`: Mô hình hồi quy tuyến tính mặc định.
  * `Ridge Regression`: Mô hình hồi quy có điều chuẩn L2 (`alpha=1.0`) giúp chống hiện tượng đa cộng tuyến và overfitting.
  * `Random Forest Regressor`: Mô hình tập hợp (ensemble) dạng cây quyết định với cấu hình 100 cây (`n_estimators=100`, `random_state=42`).
* **Đánh giá hiệu suất:** Tính toán các chỉ số sai số trực tiếp trên thang đo logarit bao gồm: Root Mean Squared Error (`log-RMSE`), Mean Absolute Error (`log-MAE`) và hệ số xác định `R2`.
* **Đóng gói kết quả:** Xuất bảng tổng hợp metrics ra file `outputs/metrics/baseline_metrics.csv` để phục vụ cho báo cáo CRISP-DM của nhóm.

## 2. Kết quả đánh giá Baseline

Dựa trên quá trình chạy thực tế, kết quả đo lường trên tập Validation thu được như sau:

| Model | Validation log-RMSE | Validation log-MAE | Validation R2 |
| :--- | :--- | :--- | :--- |
| Linear Regression | 0.1332 | 0.0955 | 0.9049 |
| Ridge Regression (alpha=1.0) | 0.1363 | 0.0973 | 0.9004 |
| Random Forest Baseline | 0.1517 | 0.0992 | 0.8767 |

**Nhận xét:**
* Các mô hình hồi quy tuyến tính (**Linear Regression** và **Ridge**) đang cho thấy sự ổn định và ưu việt rõ rệt trên tập dữ liệu dạng bảng có nhiều biến phân loại (đã one-hot) này so với mô hình cây (Random Forest).
* Mô hình **Linear Regression** đang dẫn đầu về hiệu suất với mức sai số `log-RMSE` đạt **0.1332** và giải thích được khoảng 90.49% sự biến thiên của giá nhà trên tập validation. 
* Chỉ số **`log-RMSE: 0.1332`** sẽ chính thức trở thành mốc Benchmark dự án. Mục tiêu của nhiệm vụ tiếp theo (Thiết kế PyTorch MLP) là phải xây dựng được cấu trúc mạng và chiến lược huấn luyện phù hợp sao cho đạt được mức sai số tiệm cận hoặc thấp hơn con số này.

In [8]:
import os

output_dir = "../outputs/metrics"
os.makedirs(output_dir, exist_ok=True)  # Tạo thư mục nếu chưa có

results_df.to_csv(f"{output_dir}/baseline_metrics.csv", index=False)

print("Đã xuất kết quả ra file baseline_metrics.csv")

Đã xuất kết quả ra file baseline_metrics.csv
